In [1]:
import pandas as pd
import requests

# OpenAQ gives hourly air quality readings
# We ask for PM2.5 measurements from London sensors
# No API key needed

url = "https://api.openaq.org/v2/measurements"

params = {
    "parameter": "pm25",      # particle matter 2.5 — common air quality measure
    "limit": 1000,            # how many readings
    "page": 1,
    "country": "US",          # change to any country
    "order_by": "datetime",
    "sort": "desc"            # newest first
}

response = requests.get(url, params=params)
print("Status code:", response.status_code)

Status code: 410


In [2]:
raw = response.json()

# The actual data is inside the 'results' key
readings = raw['results']
df = pd.DataFrame(readings)

print("Shape:", df.shape)
df.head()

KeyError: 'results'

In [ ]:
# OpenAQ returns some columns as nested dictionaries
# For example the 'date' column contains {'utc': '...', 'local': '...'}
# You need to flatten these

# Extract the UTC datetime
df['datetime'] = df['date'].apply(lambda x: x['utc'])

# Extract coordinates if nested
if 'coordinates' in df.columns:
    df['lat'] = df['coordinates'].apply(
        lambda x: x['latitude'] if x else None
    )
    df['lon'] = df['coordinates'].apply(
        lambda x: x['longitude'] if x else None
    )

# Keep only the columns you need
df_clean = df[['datetime', 'value', 'unit', 'parameter', 
               'location', 'city', 'country']].copy()

df_clean['datetime'] = pd.to_datetime(df_clean['datetime'])
df_clean = df_clean.sort_values('datetime')

print(df_clean.shape)
df_clean.head()